In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
model_name = "AryanDabad/trained_gemma_3_1b_it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

In [ ]:
def generate_response(prompt, max_tokens=200, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

In [ ]:
def summarize_history(history):
    if not history:
        return "No conversation history."

    prompt = (
        "### Instruction:\nSummarize the following conversation history concisely while preserving key details.\n\n"
        f"### Input:\n{history}\n\n### Response:\n"
    )
    try:
        return generate_response(prompt, max_tokens=200, temperature=0.3)
    except Exception as e:
        print(f"Error during summarization: {e}")
        return "Error in summarization."

In [ ]:
def get_chatbot_response(summary, user_input):
    prompt = (
        "### Instruction:\nBased on the summarized conversation history and the current user input, generate a meaningful response.\n\n"
        f"### Input:\nConversation Summary: {summary}\nUser Input: {user_input}\n\n### Response:\n"
    )
    try:
        return generate_response(prompt, max_tokens=150, temperature=0.7)
    except Exception as e:
        print(f"Error during response generation: {e}")
        return "Error in generating response."

In [ ]:
def chatbot():
    print("Welcome to the AI Chatbot! Type 'exit' to quit.")
    history = []

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Chatbot: Goodbye!")
            break

        history.append(f"User: {user_input}")
        summary = summarize_history("\n".join(history))
        response = get_chatbot_response(summary, user_input)
        print(f"Chatbot: {response}")
        history.append(f"Chatbot: {response}")

In [ ]:
chatbot()
